In [0]:
%sql
WITH customer_spending AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS total_spend
    FROM workspace.default.customers c
    JOIN workspace.default.orders o
        ON c.customer_id = o.customer_id
    JOIN workspace.default.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY
        c.customer_id,
        c.customer_name
)

SELECT
    customer_id,
    customer_name,
    ROUND(total_spend, 2) AS total_spend,
    RANK() OVER (
        ORDER BY total_spend DESC
    ) AS customer_rank
FROM customer_spending
ORDER BY customer_rank;

customer_id,customer_name,total_spend,customer_rank
C0494,Danielle Gonzalez,754021.19,1
C0347,Adam Taylor,687435.63,2
C0028,Rachel Mitchell,663253.96,3
C0083,William Tran,622024.19,4
C0022,David Caldwell,614612.0,5
C0017,Sherry Decker,602120.25,6
C0431,Michael Cooper,589441.33,7
C0277,Wanda Perez,579255.89,8
C0422,Rachel Knox,568221.66,9
C0019,Joseph Obrien,563139.3,10


In [0]:
%sql
WITH customer_spending AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS total_spend
    FROM workspace.default.customers c
    JOIN workspace.default.orders o
        ON c.customer_id = o.customer_id
    JOIN workspace.default.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY
        c.customer_id,
        c.customer_name
)

SELECT
    customer_id,
    customer_name,
    ROUND(total_spend, 2) AS total_spend,
    DENSE_RANK() OVER (
        ORDER BY total_spend DESC
    ) AS spending_rank
FROM customer_spending
ORDER BY spending_rank;

customer_id,customer_name,total_spend,spending_rank
C0494,Danielle Gonzalez,754021.19,1
C0347,Adam Taylor,687435.63,2
C0028,Rachel Mitchell,663253.96,3
C0083,William Tran,622024.19,4
C0022,David Caldwell,614612.0,5
C0017,Sherry Decker,602120.25,6
C0431,Michael Cooper,589441.33,7
C0277,Wanda Perez,579255.89,8
C0422,Rachel Knox,568221.66,9
C0019,Joseph Obrien,563139.3,10


In [0]:
%sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue
    FROM workspace.default.orders o
    JOIN workspace.default.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY DATE_TRUNC('month', o.order_date)
)

SELECT
    month,
    ROUND(revenue, 2) AS revenue
FROM monthly_revenue
ORDER BY month;

month,revenue
null,3530775.21
2025-08-01T00:00:00.000Z,3581194.53
2025-09-01T00:00:00.000Z,4304049.59
2025-10-01T00:00:00.000Z,5845699.83
2025-11-01T00:00:00.000Z,3928351.46
2025-12-01T00:00:00.000Z,5748819.67
2026-01-01T00:00:00.000Z,4728350.9
2026-02-01T00:00:00.000Z,7165603.64
2026-03-01T00:00:00.000Z,5666008.79
2026-04-01T00:00:00.000Z,4475171.52


In [0]:
%sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue
    FROM workspace.default.orders o
    JOIN workspace.default.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY DATE_TRUNC('month', o.order_date)
)

SELECT
    month,
    ROUND(revenue, 2) AS monthly_revenue,

    ROUND(
        SUM(revenue) OVER (
            ORDER BY month
        ),
        2
    ) AS running_revenue

FROM monthly_revenue
ORDER BY month;

month,monthly_revenue,running_revenue
null,3530775.21,3530775.21
2025-08-01T00:00:00.000Z,3581194.53,7111969.74
2025-09-01T00:00:00.000Z,4304049.59,1.141601932E7
2025-10-01T00:00:00.000Z,5845699.83,1.726171915E7
2025-11-01T00:00:00.000Z,3928351.46,2.119007061E7
2025-12-01T00:00:00.000Z,5748819.67,2.693889028E7
2026-01-01T00:00:00.000Z,4728350.9,3.166724117E7
2026-02-01T00:00:00.000Z,7165603.64,3.883284482E7
2026-03-01T00:00:00.000Z,5666008.79,4.44988536E7
2026-04-01T00:00:00.000Z,4475171.52,4.897402513E7


In [0]:
%sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue
    FROM workspace.default.orders o
    JOIN workspace.default.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY DATE_TRUNC('month', o.order_date)
)

SELECT
    month,
    ROUND(revenue, 2) AS monthly_revenue,

    ROUND(
        AVG(revenue) OVER (
            ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS three_month_average

FROM monthly_revenue
ORDER BY month;

month,monthly_revenue,three_month_average
null,3530775.21,3530775.21
2025-08-01T00:00:00.000Z,3581194.53,3555984.87
2025-09-01T00:00:00.000Z,4304049.59,3805339.77
2025-10-01T00:00:00.000Z,5845699.83,4576981.31
2025-11-01T00:00:00.000Z,3928351.46,4692700.29
2025-12-01T00:00:00.000Z,5748819.67,5174290.32
2026-01-01T00:00:00.000Z,4728350.9,4801840.67
2026-02-01T00:00:00.000Z,7165603.64,5880924.74
2026-03-01T00:00:00.000Z,5666008.79,5853321.11
2026-04-01T00:00:00.000Z,4475171.52,5768927.98


In [0]:
%sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue
    FROM workspace.default.orders o
    JOIN workspace.default.order_items oi
        ON o.order_id = oi.order_id
    GROUP BY DATE_TRUNC('month', o.order_date)
),

previous_month AS (
    SELECT
        month,
        revenue,

        LAG(revenue) OVER (
            ORDER BY month
        ) AS previous_revenue

    FROM monthly_revenue
)

SELECT
    month,
    ROUND(revenue, 2) AS revenue,
    ROUND(previous_revenue, 2) AS previous_month_revenue,

    ROUND(
        ((revenue - previous_revenue)
        / previous_revenue) * 100,
        2
    ) AS growth_percent

FROM previous_month
ORDER BY month;

month,revenue,previous_month_revenue,growth_percent
null,3530775.21,null,null
2025-08-01T00:00:00.000Z,3581194.53,3530775.21,1.43
2025-09-01T00:00:00.000Z,4304049.59,3581194.53,20.18
2025-10-01T00:00:00.000Z,5845699.83,4304049.59,35.82
2025-11-01T00:00:00.000Z,3928351.46,5845699.83,-32.8
2025-12-01T00:00:00.000Z,5748819.67,3928351.46,46.34
2026-01-01T00:00:00.000Z,4728350.9,5748819.67,-17.75
2026-02-01T00:00:00.000Z,7165603.64,4728350.9,51.55
2026-03-01T00:00:00.000Z,5666008.79,7165603.64,-20.93
2026-04-01T00:00:00.000Z,4475171.52,5666008.79,-21.02


In [0]:
%sql
WITH customer_value AS (
    SELECT
        c.customer_id,
        c.customer_name,

        COUNT(DISTINCT o.order_id) AS total_orders,

        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS lifetime_value

    FROM workspace.default.customers c

    JOIN workspace.default.orders o
        ON c.customer_id = o.customer_id

    JOIN workspace.default.order_items oi
        ON o.order_id = oi.order_id

    GROUP BY
        c.customer_id,
        c.customer_name
)

SELECT
    customer_id,
    customer_name,
    total_orders,
    ROUND(lifetime_value, 2) AS lifetime_value,

    DENSE_RANK() OVER (
        ORDER BY lifetime_value DESC
    ) AS lifetime_value_rank

FROM customer_value
ORDER BY lifetime_value_rank;

customer_id,customer_name,total_orders,lifetime_value,lifetime_value_rank
C0494,Danielle Gonzalez,7,754021.19,1
C0347,Adam Taylor,4,687435.63,2
C0028,Rachel Mitchell,4,663253.96,3
C0083,William Tran,3,622024.19,4
C0022,David Caldwell,3,614612.0,5
C0017,Sherry Decker,6,602120.25,6
C0431,Michael Cooper,4,589441.33,7
C0277,Wanda Perez,4,579255.89,8
C0422,Rachel Knox,6,568221.66,9
C0019,Joseph Obrien,3,563139.3,10


In [0]:
%sql 
WITH product_sales AS (
    SELECT
        p.product_id,
        p.product_name,

        SUM(
            oi.quantity * oi.unit_price
            * (1 - oi.discount_percent / 100)
        ) AS revenue

    FROM workspace.default.products p

    JOIN workspace.default.order_items oi
        ON p.product_id = oi.product_id

    GROUP BY
        p.product_id,
        p.product_name
)

SELECT
    product_id,
    product_name,
    ROUND(revenue, 2) AS revenue,

    RANK() OVER (
        ORDER BY revenue DESC
    ) AS product_rank

FROM product_sales
ORDER BY product_rank;

product_id,product_name,revenue,product_rank
P0292,Bedsheet,569771.28,1
P0446,Chair,556068.43,2
P0329,T-Shirt,505647.87,3
P0095,Magazine,472494.57,4
P0338,Bedsheet,471919.37,5
P0097,Novel,467562.19,6
P0052,Headphones,463422.32,7
P0422,Shoes,447600.48,8
P0065,Chair,446094.49,9
P0372,Lamp,443497.56,10
